In [4]:
# Bulk preprocessing

import pandas as pd
import os
import glob

save_path = (
    "/lakehouse/default/Files/data/"
    "processed/deals/bulk_processed.csv"
)

os.makedirs(
    "/lakehouse/default/Files/data/processed/deals",
    exist_ok=True
)

def clean_bulk(files, source):

    data = []

    for file in files:

        try:
            df = pd.read_csv(file)

        except pd.errors.EmptyDataError:
            print(
                "Skipped empty file:",
                os.path.basename(file)
            )
            continue

        if df.empty:
            continue

        # ------------------------
        # Clean column names
        # ------------------------

        df.columns = (
            df.columns
            .str.strip()
            .str.replace('"', '', regex=False)
            .str.replace('\ufeff', '', regex=False)
        )

        df.columns = [
            col.replace("Date ", "Date")
            for col in df.columns
        ]

        # ------------------------
        # Rename columns
        # ------------------------

        if source == "NSE":

            df = df.rename(columns={
                "Date": "date",
                "Symbol": "symbol",
                "Security Name": "security_name",
                "Client Name": "client_name",
                "Buy / Sell": "deal_type",
                "Quantity Traded": "quantity",
                "Trade Price / Wght. Avg. Price": "price"
            })

        else:

            df = df.rename(columns={
                "Deal Date": "date",
                "Deal_Date": "date",
                "Security Code": "symbol",
                "Security_Code": "symbol",
                "Company": "security_name",
                "Client Name": "client_name",
                "Client_Name": "client_name",
                "Deal Type": "deal_type",
                "Deal_Type": "deal_type",
                "Quantity": "quantity",
                "Price": "price"
            })

        keep_cols = [
            "date",
            "symbol",
            "security_name",
            "client_name",
            "deal_type",
            "quantity",
            "price"
        ]

        df = df[keep_cols]

        # ------------------------
        # Trim text columns
        # ------------------------

        text_cols = [
            "symbol",
            "security_name",
            "client_name",
            "deal_type"
        ]

        for col in text_cols:

            df[col] = (
                df[col]
                .astype(str)
                .str.strip()
            )

        # ------------------------
        # Quantity
        # ------------------------

        df["quantity"] = (
            df["quantity"]
            .astype(str)
            .str.replace(",", "", regex=False)
        )

        df["quantity"] = pd.to_numeric(
            df["quantity"],
            errors="coerce"
        )

        # ------------------------
        # Price
        # ------------------------

        df["price"] = pd.to_numeric(
            df["price"],
            errors="coerce"
        )

        # ------------------------
        # Deal Type
        # ------------------------

        df["deal_type"] = (
            df["deal_type"]
            .astype(str)
            .str.upper()
            .replace({
                "B": "BUY",
                "BUY": "BUY",
                "P": "BUY",
                "S": "SELL",
                "SELL": "SELL",
                "SOLD": "SELL"
            })
        )

        # ------------------------
        # Date
        # ------------------------

        df["date"] = pd.to_datetime(
            df["date"],
            errors="coerce",
            dayfirst=True
        )

        # ------------------------
        # Source
        # ------------------------

        df["source"] = source

        data.append(df)

    if len(data) == 0:
        return pd.DataFrame()

    return pd.concat(
        data,
        ignore_index=True
    )

# ======================================
# Read all data
# ======================================

final_df = pd.concat([

    clean_bulk(
        glob.glob(
            "/lakehouse/default/Files/data/raw/nse_bulk_hist/*.csv"
        ),
        "NSE"
    ),

    clean_bulk(
        glob.glob(
            "/lakehouse/default/Files/data/raw/nse_bulk_inc/*.csv"
        ),
        "NSE"
    ),

    clean_bulk(
        glob.glob(
            "/lakehouse/default/Files/data/raw/bse_bulk_hist/*.csv"
        ),
        "BSE"
    ),

    clean_bulk(
        glob.glob(
            "/lakehouse/default/Files/data/raw/bse_bulk_inc/*.csv"
        ),
        "BSE"
    )

], ignore_index=True)

# ======================================
# Cleaning
# ======================================

final_df = final_df.dropna(
    subset=[
        "date",
        "security_name",
        "deal_type"
    ]
)

# Remove exact duplicates

final_df = final_df.drop_duplicates()

# Remove zero/negative prices
# Keep NULL because NSE doesn't provide price for many deals

final_df = final_df[
    final_df["price"].isna() |
    (final_df["price"] > 0)
]

# Sort

final_df = final_df.sort_values(
    [
        "date",
        "symbol",
        "client_name"
    ]
)

# ======================================
# Save
# ======================================

final_df.to_csv(
    save_path,
    index=False
)

print("SUCCESS")
print("Rows:", len(final_df))
print("Saved:", save_path)

StatementMeta(, c96243e4-9edf-4704-85cf-6e6338d20c4e, 6, Finished, Available, Finished, False)

/tmp/ipykernel_6514/4093767045.py:161: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(


SUCCESS
Rows: 515492
Saved: /lakehouse/default/Files/data/processed/deals/bulk_processed.csv


In [5]:
bulk_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(
        "Files/data/processed/deals/bulk_processed.csv"
    )
)

spark.sql(
    "DROP TABLE IF EXISTS bulk_daily"
)

(
    bulk_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("bulk_daily")
)

print("SUCCESS")
print("Rows:", bulk_df.count())

bulk_df.printSchema()

display(
    bulk_df.limit(10)
)

StatementMeta(, c96243e4-9edf-4704-85cf-6e6338d20c4e, 7, Finished, Available, Finished, False)

SUCCESS
Rows: 515492
root
 |-- date: date (nullable = true)
 |-- symbol: string (nullable = true)
 |-- security_name: string (nullable = true)
 |-- client_name: string (nullable = true)
 |-- deal_type: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- source: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 76be4c87-c308-450b-bb98-e4db9ac8de50)